# Synthetic Spikes Toy Task

A fast, zero-download task for quickly testing and debugging GLIF RSNN training.

**Task:** Classify synthetic spike patterns into 4 classes based on firing statistics.

| Class | Pattern |
|-------|---------|
| 0 | Low firing rate (~5 Hz) |
| 1 | High firing rate (~25 Hz) |
| 2 | Early burst (first half active) |
| 3 | Late burst (second half active) |

In [ ]:
import torch
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from btorch.models import environ, functional
from btorch.models.init import uniform_v_

from src.model import SingleLayerGLIFRSNN
from src.loss import CombinedLoss, LossConfig
from tutorials.synthetic_spikes import get_dataloaders, get_task_defaults

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Load Data

In [ ]:
defaults = get_task_defaults()
print("Task defaults:", defaults)

class Config:
    pass

config = Config()
for k, v in defaults.items():
    setattr(config, k, v)

train_loader, test_loader, input_dim, output_dim, T = get_dataloaders(config)

print(f"\nData loaded:")
print(f"  Input dim: {input_dim}")
print(f"  Output dim: {output_dim}")
print(f"  Timesteps: {T}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")

## 2. Visualize Synthetic Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for class_idx in range(4):
    # Find a sample of this class
    for x_batch, y_batch in train_loader:
        mask = y_batch == class_idx
        if mask.any():
            x_sample = x_batch[:, mask, :][:, 0, :].cpu()
            break
    
    ax = axes[class_idx]
    for neuron_idx in range(input_dim):
        spike_times = torch.where(x_sample[:, neuron_idx] > 0)[0]
        ax.scatter(spike_times.numpy(), [neuron_idx] * len(spike_times), 
                   c='black', s=5)
    
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Neuron Index')
    ax.set_title(f'Class {class_idx}')
    ax.set_xlim(0, T)

plt.suptitle('Synthetic Spike Patterns')
plt.tight_layout()
plt.show()

## 3. Quick Smoke Test

In [ ]:
model = SingleLayerGLIFRSNN(
    input_dim=input_dim,
    output_dim=output_dim,
    n_neuron=defaults['n_neuron'],
    dt=defaults['dt'],
).to(device)

# Initialize state
functional.init_net_state(model.rnn, batch_size=config.batch_size, device=device)
uniform_v_(model.neuron, set_reset_value=True)

optimizer = torch.optim.Adam(model.parameters(), lr=defaults['lr'])
loss_fn = CombinedLoss(LossConfig(), dt=defaults['dt'])

# Fast training loop
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (x, target) in enumerate(train_loader):
        x, target = x.to(device), target.to(device)
        
        functional.reset_net(model.rnn, batch_size=x.shape[1])
        optimizer.zero_grad()
        
        with environ.context(dt=defaults['dt']):
            output, states = model(x)
        
        loss, _ = loss_fn(output, target, states, T)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)
    
    acc = 100. * correct / total
    print(f"Epoch {epoch+1}/{num_epochs}: loss={total_loss/len(train_loader):.4f}, acc={acc:.2f}%")

## 4. Evaluate

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for x, target in test_loader:
        x, target = x.to(device), target.to(device)
        
        functional.reset_net(model.rnn, batch_size=x.shape[1])
        
        with environ.context(dt=defaults['dt']):
            output, states = model(x)
        
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)

print(f"Test accuracy: {100. * correct / total:.2f}%")